In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from mpl_toolkits.mplot3d import Axes3D
from pathlib import Path

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.dpi"] = 300
plt.rcParams["font.family"] = "serif"
plt.rcParams["axes.titleweight"] = "bold"

charts = Path("charts")
charts.mkdir(exist_ok=True)

returns = pd.read_csv("india_equity_returns.csv", index_col="Date", parse_dates=True)
prices = pd.read_csv("india_equity_prices.csv", index_col="Date", parse_dates=True)
sim = pd.read_csv("portfolio_simulations.csv")
metrics = pd.read_csv("portfolio_metrics.csv")
weights = pd.read_csv("portfolio_weights.csv", index_col=0)

corr = returns.corr()
cov = returns.cov()
mean_ret = returns.mean()
vol = returns.std()

n_assets = len(returns.columns)

# correlation heatmap
plt.figure(figsize=(10,8))
sns.heatmap(corr, annot=True, cmap="coolwarm")
plt.title("Asset Correlation Heatmap (%s Assets)" % n_assets)
plt.tight_layout()
plt.savefig(charts/"correlation_heatmap.png")
plt.close()

# covariance heatmap
plt.figure(figsize=(10,8))
sns.heatmap(cov, cmap="viridis")
plt.title("Asset Covariance Matrix (%s Assets)" % n_assets)
plt.tight_layout()
plt.savefig(charts/"covariance_heatmap.png")
plt.close()

# normalized prices
plt.figure(figsize=(10,6))
(prices / prices.iloc[0]).plot()
plt.title("Normalized Price Performance (%s Stocks)" % n_assets)
plt.ylabel("Normalized Price")
plt.tight_layout()
plt.savefig(charts/"normalized_prices.png")
plt.close()

# return distributions
plt.figure(figsize=(10,6))
for c in returns.columns:
    plt.hist(returns[c], bins=50, alpha=0.4)
plt.title("Distribution of Asset Returns (%s Stocks)" % n_assets)
plt.tight_layout()
plt.savefig(charts/"return_distributions.png")
plt.close()

# find optimal portfolios
max_sharpe = sim.loc[sim["Sharpe"].idxmax()]
min_vol = sim.loc[sim["Volatility"].idxmin()]

rf = 0.05

# efficient frontier
plt.figure(figsize=(10,6))
scatter = plt.scatter(sim["Volatility"], sim["Return"], c=sim["Sharpe"], cmap="viridis")
plt.colorbar(scatter,label="Sharpe Ratio")

plt.scatter(max_sharpe["Volatility"], max_sharpe["Return"],
            color="red", s=120, marker="*", label="Max Sharpe")

plt.scatter(min_vol["Volatility"], min_vol["Return"],
            color="blue", s=120, marker="D", label="Min Variance")

x = np.linspace(0, sim["Volatility"].max(), 100)
cml = rf + (max_sharpe["Return"] - rf) / max_sharpe["Volatility"] * x
plt.plot(x, cml, linestyle="--", label="Capital Market Line")

title = "Efficient Frontier | Max Sharpe %.3f | Min Vol %.3f" % (
    max_sharpe["Sharpe"], min_vol["Volatility"]
)

plt.xlabel("Volatility")
plt.ylabel("Return")
plt.title(title)
plt.legend()
plt.tight_layout()
plt.savefig(charts/"efficient_frontier_2d.png")
plt.close()

# efficient frontier 3d
fig = plt.figure(figsize=(10,7))
ax = fig.add_subplot(111, projection="3d")

p = ax.scatter(sim["Volatility"], sim["Return"], sim["Sharpe"],
               c=sim["Sharpe"], cmap="viridis")

fig.colorbar(p, ax=ax, label="Sharpe Ratio")

ax.scatter(max_sharpe["Volatility"], max_sharpe["Return"],
           max_sharpe["Sharpe"], color="red", s=120)

ax.scatter(min_vol["Volatility"], min_vol["Return"],
           min_vol["Sharpe"], color="blue", s=120)

ax.set_xlabel("Volatility")
ax.set_ylabel("Return")
ax.set_zlabel("Sharpe")

ax.set_title("3D Portfolio Risk–Return–Sharpe (Max Sharpe %.3f)" % max_sharpe["Sharpe"])

plt.tight_layout()
plt.savefig(charts/"efficient_frontier_3d.png")
plt.close()

# mean returns
plt.figure(figsize=(10,6))
mean_ret.plot(kind="bar")
plt.title("Average Asset Returns (%s Stocks)" % n_assets)
plt.ylabel("Mean Return")
plt.tight_layout()
plt.savefig(charts/"mean_returns_bar.png")
plt.close()

# portfolio metrics
plt.figure(figsize=(10,6))
metrics.set_index("Portfolio").plot(kind="bar")

best = metrics.loc[metrics["Sharpe"].idxmax()]

plt.title("Portfolio Metrics | Best Sharpe: %.3f (%s)" %
          (best["Sharpe"], best["Portfolio"]))

plt.tight_layout()
plt.savefig(charts/"portfolio_metrics.png")
plt.close()

# asset risk return scatter
plt.figure(figsize=(10,6))
plt.scatter(vol, mean_ret)

for ticker in mean_ret.index:
    plt.annotate(ticker,(vol[ticker],mean_ret[ticker]))

plt.xlabel("Volatility")
plt.ylabel("Return")
plt.title("Asset Risk–Return Profile (%s Assets)" % n_assets)
plt.tight_layout()
plt.savefig(charts/"asset_risk_return_scatter.png")
plt.close()

# cumulative portfolio returns
equal_w = weights["Equal_Weight"].values
opt_w = weights["Optimized_Weight"].values

port_equal = (returns @ equal_w).cumsum()
port_opt = (returns @ opt_w).cumsum()

plt.figure(figsize=(10,6))
plt.plot(port_equal,label="Equal Weight")
plt.plot(port_opt,label="Optimized")
plt.legend()

plt.title("Portfolio Cumulative Returns | Optimized Sharpe %.3f" %
          max_sharpe["Sharpe"])

plt.ylabel("Cumulative Return")
plt.tight_layout()
plt.savefig(charts/"portfolio_cumulative_returns.png")
plt.close()

# portfolio weights
plt.figure(figsize=(10,6))
weights.plot(kind="bar")

plt.title("Portfolio Weight Allocation (%s Assets)" % n_assets)

plt.ylabel("Weight")
plt.tight_layout()
plt.savefig(charts/"portfolio_weights.png")
plt.close()

# sharpe comparison
plt.figure(figsize=(8,6))
metrics.set_index("Portfolio")["Sharpe"].plot(kind="bar")

best = metrics.loc[metrics["Sharpe"].idxmax()]

plt.title("Sharpe Ratio Comparison | Best: %.3f (%s)" %
          (best["Sharpe"], best["Portfolio"]))

plt.ylabel("Sharpe Ratio")
plt.tight_layout()
plt.savefig(charts/"sharpe_comparison.png")
plt.close()

<Figure size 3000x1800 with 0 Axes>

<Figure size 3000x1800 with 0 Axes>

<Figure size 3000x1800 with 0 Axes>